# Notebook 03 — Empirical Analysis

Phase 3 of the "Stablecoins vs. SWIFT" project. Tests four
hypotheses (H1–H4) against the master datasets built in Phase 2B.

**Structure:**
- §0 Setup: imports, seed, style, data loads, smoke test
- §1 H1 Metcalfe's Law (log-log OLS, ADF, Wald, structural break)
- §2 H3 Market concentration (HHI trend, event decomposition)
- §3 H4 Cost friction (paired tests at $200 and $10,000)
- §4 H2 Diffusion (pooled → country FE → two-way FE ladder)

All methodology decisions are logged in
`docs/PHASE_3_DECISIONS.md` (D-01 through D-10). Each section
below references the relevant decision IDs.

**Execution order rationale:** H1 opens the notebook because
Metcalfe's Law is the foundational network-effects claim that
the rest of the analysis builds on. H2 closes the notebook
because its specification ladder is the most complex.

## §0 — Setup

Imports, random seed, matplotlib style, data loads, assertions,
linearmodels smoke test. All setup lives in §0; no imports or
seeds later in the notebook.

In [1]:
# Standard library
import random
from pathlib import Path

# Numerical and data
import numpy as np
import pandas as pd

# Econometrics
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from linearmodels.panel import PanelOLS

# Stats
from scipy import stats

# Plotting
import matplotlib.pyplot as plt
import matplotlib as mpl

In [2]:
# D-06: global random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
# D-05: figure style conventions
mpl.rcParams["figure.dpi"] = 100         # screen display
mpl.rcParams["savefig.dpi"] = 300        # saved output
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.grid"] = False
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

# Three-color palette for all Phase 3 figures
PALETTE = {
    "primary":   "#1f77b4",   # tab:blue
    "secondary": "#ff7f0e",   # tab:orange
    "tertiary":  "#2ca02c",   # tab:green
    "muted":     "#7f7f7f",   # tab:gray (for annotations)
}

In [4]:
# D-08: no hardcoded paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = REPO_ROOT / "data" / "03_processed"
FIG_DIR = REPO_ROOT / "outputs" / "figures"
TBL_DIR = REPO_ROOT / "outputs" / "tables"

assert DATA_DIR.exists(), f"DATA_DIR missing: {DATA_DIR}"
assert FIG_DIR.exists(), f"FIG_DIR missing: {FIG_DIR}"
assert TBL_DIR.exists(), f"TBL_DIR missing: {TBL_DIR}"
print(f"REPO_ROOT: {REPO_ROOT}")

REPO_ROOT: C:\dev\ine


In [5]:
# Load all four master datasets with expected shapes (from
# Phase 2C Task A verification).
h1 = pd.read_csv(DATA_DIR / "h1_network_effects.csv",
                 parse_dates=["date"])
h2 = pd.read_csv(DATA_DIR / "h2_diffusion_dataset.csv")
h3 = pd.read_csv(DATA_DIR / "h3_concentration.csv",
                 parse_dates=["date"])
h4 = pd.read_csv(DATA_DIR / "h4_infrastructure_cost.csv")

# Shape assertions match Phase 2C baseline exactly
assert h1.shape == (4384, 4), f"h1 shape mismatch: {h1.shape}"
assert h2.shape == (861, 23),  f"h2 shape mismatch: {h2.shape}"
assert h3.shape == (72, 7),    f"h3 shape mismatch: {h3.shape}"
assert h4.shape == (72, 12),   f"h4 shape mismatch: {h4.shape}"

# Content assertions
assert set(h1["asset"].unique()) == {"USDC", "USDT"}
assert h1["date"].min() == pd.Timestamp("2020-01-01")
assert h1["date"].max() == pd.Timestamp("2025-12-31")
assert h3["date"].min() == pd.Timestamp("2020-01-01")
assert h3["date"].max() == pd.Timestamp("2025-12-01")
assert h2["year"].min() == 2020
assert h2["year"].max() == 2025
assert h4["month"].min() == "2020-01"
assert h4["month"].max() == "2025-12"

print("All four master datasets loaded.")
print(f"  h1: {h1.shape}  assets: {sorted(h1['asset'].unique())}")
print(f"  h2: {h2.shape}  countries: {h2['country_iso3'].nunique()}")
print(f"  h3: {h3.shape}  months: {len(h3)}")
print(f"  h4: {h4.shape}  months: {len(h4)}")

All four master datasets loaded.
  h1: (4384, 4)  assets: ['USDC', 'USDT']
  h2: (861, 23)  countries: 160
  h3: (72, 7)  months: 72
  h4: (72, 12)  months: 72


### §0.1 Smoke test — linearmodels compatibility

Day-1 infrastructure check. Verifies `linearmodels==7.0` fits a
`PanelOLS` model against the current `pandas` / `numpy` versions.
If this fails, STOP and escalate before doing any H2 analysis
work. This is not a real regression — it's a handshake with the
dependency.

In [6]:
# Smoke test: PanelOLS must return non-null parameters on a
# trivial fit. Uses h2's MultiIndex (country_iso3, year).
_smoke = h2.dropna(subset=["gdp_per_capita_usd"]).set_index(
    ["country_iso3", "year"]
)
_smoke_fit = PanelOLS(
    _smoke["adoption_percentile"],
    sm.add_constant(_smoke[["gdp_per_capita_usd"]]),
    entity_effects=True,
).fit()
assert _smoke_fit.params.notna().all(), \
    "linearmodels smoke test failed — PanelOLS returned NaN params"
print("linearmodels smoke test: OK")
print(f"  observations: {_smoke_fit.nobs}")
print(f"  entities: {_smoke_fit.entity_info.total}")
del _smoke, _smoke_fit

linearmodels smoke test: OK
  observations: 859
  entities: 160.0


## §1 — H1 Metcalfe's Law

Tests whether stablecoin transfer activity scales with active
addresses in log-log space. Primary spec: log-log OLS with
Newey-West HAC standard errors, per asset. Wald tests at β = 1
(linear) and β = 2 (strict Metcalfe). ADF stationarity pre-check.
Pre/post structural break at 2022-11-11 (FTX).

**Decisions invoked:** D-03 (structural break date),
D-06 (seed), D-09 (HAC maxlags = 12 for daily data).

**Subsections (to be implemented in Prompt 3):**
- §1.1 Log transforms and positivity checks
- §1.2 ADF stationarity tests (levels and first differences)
- §1.3 Log-log OLS per asset, full window
- §1.4 Wald tests: H0: β=1, H0: β=2
- §1.5 Pre/post structural break sub-samples

## §2 — H3 Market Concentration

Monthly HHI across stablecoins. Primary spec: OLS trend on
`time_index` with Newey-West HAC SE. Headline split at Dec 2022
(per D-01) with Jun 2022 robustness. `hhi_top5` as secondary
robustness.

**Decisions invoked:** D-01 (post-crisis cutoff),
D-05 (figure style), D-07 (output naming), D-09 (HAC maxlags = 4
for monthly data).

**Subsections (to be implemented in Prompt 4):**
- §2.1 HHI time series figure with event annotations
- §2.2 Structural event table
- §2.3 OLS trend, full window
- §2.4 OLS trend, post-Dec-2022 split (headline)
- §2.5 OLS trend, post-Jun-2022 split (robustness)
- §2.6 hhi_top5 robustness

## §3 — H4 Cost Friction

Monthly paired differences between legacy remittance cost and
on-chain fee, per rail (ETH, Tron) and per transfer size ($200,
$10,000). Test implemented as OLS of monthly differences on a
constant with HAC SE (D-10). Tron side uses `tron_median_fee_usd`
(D-02). Headline uses `legacy_flat_fee = 0`; $3.50 sensitivity
row reported underneath.

**Decisions invoked:** D-02 (paired-test design),
D-05 (figure style), D-07 (output naming), D-09 (HAC maxlags = 4),
D-10 (test implementation).

**Subsections (to be implemented in Prompt 5):**
- §3.1 Monthly fee time series figure (ETH, Tron, legacy)
- §3.2 Cost comparison bar chart at $200 and $10,000
- §3.3 Paired-test table: 4 rails/sizes × 2 flat-fee scenarios
- §3.4 ETH-Tron crossover annotation (post-Dencun 2025 months)

## §4 — H2 Diffusion & Institutional Gaps

Country-year panel of Chainalysis adoption index on macro
controls and financial-inclusion baseline. Five-spec ladder per
D-04: pooled OLS → country FE → two-way FE with
`baseline × post_2022` interaction → two-way FE excluding
forward-filled rows → two-way FE with
`baseline_year == 2024` interaction. Country-clustered SE
throughout. Specs 3–5 exclude 8 single-year countries.

**Decisions invoked:** D-04 (specification ladder),
D-05 (figure style), D-07 (output naming), D-08 (hygiene).

**Subsections (to be implemented in Prompt 6):**
- §4.1 Descriptive: adoption distribution by year/region
- §4.2 Pooled OLS with country-clustered SE
- §4.3 Country FE
- §4.4 Two-way FE with baseline × post_2022 interaction
- §4.5 Robustness: exclude forward-filled 2025 rows
- §4.6 Robustness: interact baseline with
  baseline_year == 2024

## §5 — Phase 3 Exit Checklist

To be filled in at Phase 3 close. Mirrors the Phase 2B exit
criteria pattern. Items include:
- [ ] All four hypotheses have completed analysis sections
- [ ] All figures saved to `outputs/figures/` at 300 DPI
- [ ] All tables saved to `outputs/tables/` as CSV + LaTeX
- [ ] Notebook runs top-to-bottom on a fresh kernel
- [ ] Every methodology decision logged in
  `docs/PHASE_3_DECISIONS.md`
- [ ] Global random seed set; no non-deterministic outputs
- [ ] No hardcoded paths; all paths derived from `REPO_ROOT`
- [ ] All assertions pass